<a href="https://colab.research.google.com/github/DKavya8/chestxray-bias-audit/blob/main/notebooks/08_operating_point_robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np, pandas as pd, os, glob, json

In [2]:
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/team-RACK-bias-paper"
RESULTS = f"{BASE}/results"
print("BASE exists?", os.path.exists(BASE))

Mounted at /content/drive
BASE exists? True


In [3]:
!rm -rf /content/repo && git clone -q https://github.com/DKavya8/chestxray-bias-audit /content/repo
REPO = "/content/repo"
print("seed dirs:", len(glob.glob(f"{REPO}/splits/seed_*")), "(expect 10)")

seed dirs: 10 (expect 10)


In [4]:
scores = pd.read_parquet(f"{RESULTS}/densenet121_all_scores.parquet")
meta   = pd.read_csv(f"{RESULTS}/metadata_clean.csv")
print("scores:", scores.shape, "| meta:", meta.shape)

scores: (112120, 22) | meta: (112106, 24)


In [5]:
NIH_14 = ["Atelectasis","Consolidation","Infiltration","Pneumothorax","Edema","Emphysema",
          "Fibrosis","Effusion","Pneumonia","Pleural_Thickening","Cardiomegaly","Nodule","Mass","Hernia"]
sc = (scores[['Image Index'] + NIH_14]
      .rename(columns={f: f's_{f}' for f in NIH_14})
      .rename(columns={'Image Index': 'image_index'}))
md = (meta[['image_index','patient_id','age','sex','follow_up'] + NIH_14]
      .rename(columns={f: f'y_{f}' for f in NIH_14}))
img = md.merge(sc, on='image_index', how='inner', validate='one_to_one')
G = img.sort_values(['patient_id','follow_up']).drop_duplicates('patient_id', keep='first')
G = G.rename(columns={'patient_id': 'Patient ID'})
G['Patient ID'] = G['Patient ID'].astype(str)
G['age'] = G['age'].astype(float)
G['bin10'] = ((G['age']//10)*10).astype(int).astype(str)
print("G first scan per patient:", G.shape, "| sex:", dict(G['sex'].value_counts()))

G first scan per patient: (30797, 34) | sex: {'M': np.int64(16625), 'F': np.int64(14172)}


In [6]:
thr_by_seed = json.load(open(f"{REPO}/results/group_a_densenet/thresholds_by_seed.json"))
split_ids = {}
for d in sorted(glob.glob(f"{REPO}/splits/seed_*")):
    seed = d.split('seed_')[1]
    cal  = set(pd.read_csv(f"{d}/calibration_patients.csv")['Patient ID'].astype(str))
    test = set(pd.read_csv(f"{d}/test_patients.csv")['Patient ID'].astype(str))
    split_ids[seed] = (cal, test)
print("seeds:", len(split_ids))

seeds: 10


In [7]:
def pooled_fnr(df, thr, w=None):
    w = np.ones(len(df)) if w is None else np.asarray(w, float)
    fn = pos = 0.0
    for f in NIH_14:
        y = df[f'y_{f}'].to_numpy(); s = df[f's_{f}'].to_numpy(); p = (y == 1)
        pos += (w * p).sum(); fn += (w * (p & (s < thr[f]))).sum()
    return fn / pos if pos > 0 else np.nan

def sex_gap(df, thr, wcol=None):
    isF = df['sex'].to_numpy() == 'F'
    w = df[wcol].to_numpy() if wcol else np.ones(len(df))
    return pooled_fnr(df[isF], thr, w[isF]) - pooled_fnr(df[~isF], thr, w[~isF])

def ipw_weights(df):
    d = df.copy()
    fp = d[d.sex=='F']['bin10'].value_counts(normalize=True)
    mp = d[d.sex=='M']['bin10'].value_counts(normalize=True)
    bins = sorted(set(fp.index)|set(mp.index)); fp = fp.reindex(bins,fill_value=0); mp = mp.reindex(bins,fill_value=0)
    target = (fp+mp)/2
    wF = (target/fp).to_dict(); wM = (target/mp).to_dict()
    isF = d.sex.values == 'F'
    d['w'] = np.where(isF, d['bin10'].map(wF).values, d['bin10'].map(wM).values)
    return d

In [8]:
def thr_at_sensitivity(pos_scores, target):

    return float(np.quantile(pos_scores, 1 - target))

def thresholds_for(cal, target, seed):
    if target is None:
        return thr_by_seed[seed]
    return {f: (thr_at_sensitivity(cal.loc[cal[f'y_{f}']==1, f's_{f}'].to_numpy(), target)
                if (cal[f'y_{f}']==1).sum() else 0.5) for f in NIH_14}

OPERATING_POINTS = {"youden_current": None, "sensitivity_0.80": 0.80, "sensitivity_0.90": 0.90}

In [9]:
per_split = {op: [] for op in OPERATING_POINTS}
for op, target in OPERATING_POINTS.items():
    for seed, (cal_ids, test_ids) in split_ids.items():
        cal  = G[G['Patient ID'].isin(cal_ids)]
        test = G[G['Patient ID'].isin(test_ids)]
        thr  = thresholds_for(cal, target, seed)
        Sraw = sex_gap(test, thr)
        Sipw = sex_gap(ipw_weights(test), thr, wcol='w')
        per_split[op].append(dict(seed=seed, S_raw=Sraw, S_ipw=Sipw,
                                  dS_ipw=Sraw-Sipw, pooled_fnr=pooled_fnr(test, thr)))
    d = pd.DataFrame(per_split[op])
    print(f"{op}: pooled FNR~{d.pooled_fnr.mean():.3f}  S_raw={d.S_raw.mean():+.4f}  "
          f"S_ipw={d.S_ipw.mean():+.4f}  dS_ipw={d.dS_ipw.mean():+.4f}")

youden_current: pooled FNR~0.501  S_raw=+0.0229  S_ipw=+0.0108  dS_ipw=+0.0121
sensitivity_0.80: pooled FNR~0.202  S_raw=+0.0334  S_ipw=+0.0241  dS_ipw=+0.0093
sensitivity_0.90: pooled FNR~0.101  S_raw=+0.0240  S_ipw=+0.0182  dS_ipw=+0.0058


In [10]:
B = 1000
rng = np.random.default_rng(20260823)
test_cache, pmap, thr_cache = {}, {}, {}
for seed, (cal_ids, test_ids) in split_ids.items():
    cal  = G[G['Patient ID'].isin(cal_ids)]
    test = G[G['Patient ID'].isin(test_ids)].reset_index(drop=True)
    test_cache[seed] = test; pmap[seed] = test.groupby('Patient ID').indices
    for op, target in OPERATING_POINTS.items():
        thr_cache[(op, seed)] = thresholds_for(cal, target, seed)

summary = []
for op in OPERATING_POINTS:
    boot = np.empty(B)
    for bi in range(B):
        if bi % 200 == 0: print(f"{op}: draw {bi}/{B}")
        vals = []
        for seed in split_ids:
            test = test_cache[seed]; idxmap = pmap[seed]
            pids = np.array(list(idxmap.keys()))
            samp = rng.choice(pids, size=len(pids), replace=True)
            rows_ = np.concatenate([idxmap[p] for p in samp])
            tb = test.iloc[rows_]; thr = thr_cache[(op, seed)]
            vals.append(sex_gap(tb, thr) - sex_gap(ipw_weights(tb), thr, wcol='w'))
        boot[bi] = np.mean(vals)
    pts = pd.DataFrame(per_split[op]); lo, hi = np.percentile(boot, [2.5, 97.5])
    summary.append(dict(operating_point=op, pooled_fnr=round(pts.pooled_fnr.mean(),3),
                        dS_ipw=round(pts.dS_ipw.mean(),4), ci_lower=round(lo,4),
                        ci_upper=round(hi,4), survives=bool(lo>0)))
    print(f"{op}: dS_ipw={pts.dS_ipw.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  survives={lo>0}")

youden_current: draw 0/1000
youden_current: draw 200/1000
youden_current: draw 400/1000
youden_current: draw 600/1000
youden_current: draw 800/1000
youden_current: dS_ipw=+0.0121  95% CI [+0.0101, +0.0142]  survives=True
sensitivity_0.80: draw 0/1000
sensitivity_0.80: draw 200/1000
sensitivity_0.80: draw 400/1000
sensitivity_0.80: draw 600/1000
sensitivity_0.80: draw 800/1000
sensitivity_0.80: dS_ipw=+0.0093  95% CI [+0.0078, +0.0108]  survives=True
sensitivity_0.90: draw 0/1000
sensitivity_0.90: draw 200/1000
sensitivity_0.90: draw 400/1000
sensitivity_0.90: draw 600/1000
sensitivity_0.90: draw 800/1000
sensitivity_0.90: dS_ipw=+0.0058  95% CI [+0.0048, +0.0069]  survives=True


In [11]:
final = pd.DataFrame(summary)
OUT2 = f"{RESULTS}/operating_point_robustness"; os.makedirs(OUT2, exist_ok=True)
final.to_csv(f"{OUT2}/dS_by_operating_point.csv", index=False)
print(final.to_string(index=False)); print("\nwrote", f"{OUT2}/dS_by_operating_point.csv")

 operating_point  pooled_fnr  dS_ipw  ci_lower  ci_upper  survives
  youden_current       0.501  0.0121    0.0101    0.0142      True
sensitivity_0.80       0.202  0.0093    0.0078    0.0108      True
sensitivity_0.90       0.101  0.0058    0.0048    0.0069      True

wrote /content/drive/MyDrive/team-RACK-bias-paper/results/operating_point_robustness/dS_by_operating_point.csv
